## tl;dr

마법사 35%안을 포함한 6개 직업의 보너스 조합을 유지하는 것을 권장한다. 신규 96개 전투 경로에서 성장속도 최대 격차 5.26%, 평균 판매수입 최대 격차 10.79%였다. 특화 직업 전원의 100레벨 확정 상한이나 운영 통합 완료를 의미하지 않는다.


## Context & Methods

### Key Assumptions

HP 8→10시간, 마법사 35%, 다른 직업 최대 30%, 탐색 최소 4초, 아이템 판매 최대 +20%, 기존 STR 가방 공식 유지. 성직자는 WIS/CHA. 성장·판매·편의를 한 점수로 합산하지 않는다. 접속 간격은 실제 접속 이벤트가 아닌 저장시간 기반 환산이다.

일반 Python으로 코드 셀을 순차 실행했다. Jupyter 커널 자체는 미검증이며 nbconvert/ipykernel이 설치된 환경에서 `python -m jupyter nbconvert --execute --to notebook --inplace final-class-review.ipynb`로 검증할 수 있다.


## Data

### 1. Recompute fresh held-out evidence


In [1]:
from pathlib import Path
import sys,json,pandas as pd,numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'tools/analysis/final_class_bonus_review.py').exists())
sys.path.insert(0,str(root/'tools/analysis'))
from final_class_bonus_review import review
effects,comparison,summary=review()
assert len(effects)==9600 and summary['fresh_games']==96
assert summary['app_source_unchanged'] and summary['calendar_independent_check']
print('Full combat games:',summary['fresh_games'],'Level rows:',len(effects))


Full combat games: 96 Level rows: 9600


## Results

### 2. Compare benefits and separate growth from sales


In [2]:
print(pd.DataFrame.from_dict(summary['level100_effects'],orient='index').round(4).to_string())
print(pd.DataFrame(summary['scenario_summary']).query('gap_h in [8,12] and session_min == 12').round(4).to_string(index=False))
assert np.isclose(summary['max_growth_spread20_to100_pct'],5.260083174027708)
assert np.isclose(summary['max_sale_spread20_to100_pct'],10.794547454806303)


           cap_h  raw_proc_pct  search_s  sale_bonus_pct       bag         hp         mp
CLERIC    9.0583       28.2061    4.6692         19.8750   96.8125  2713.0000  4923.6875
MAGE      9.0577       35.0000    4.6692          6.7750   96.8125  2711.0000  7035.4375
PALADIN   9.0923       24.5812    4.6612         19.9333  146.9375  2820.5625  2748.7500
RANGER    9.0685       28.0332    4.0008          6.7750   97.1875  2745.2500  4819.9375
ROGUE     9.1443       24.4246    4.0058          6.4500  147.0000  2988.5625  2654.7500
WARRIOR  10.0000       24.5746    4.6612          6.4250  146.9375  7048.7500  2744.7500
 gap_h  session_min  growth_spread_pct  sale_spread_pct fastest slowest highest_sale lowest_sale
   8.0         12.0             3.5606          10.7945    MAGE WARRIOR       CLERIC     WARRIOR
  12.0         12.0             5.2601           8.0601 WARRIOR PALADIN       CLERIC       ROGUE


### 3. Verify caps and record uncertainty


In [3]:
print(pd.DataFrame(summary['cap_by_class']).to_string(index=False))
print(pd.DataFrame(summary['stability']).round(4).to_string(index=False))
print(pd.DataFrame(summary['iid_growth_tails'])[['hero_class','hp_cap_pct','mp_threshold_pct','dex_cap_pct','cha_cap_pct']].round(4).to_string(index=False))
print('Excluded:',summary['excluded'])
assert summary['bag_formula_match'] and summary['raw_proc_formula_match']


hero_class  samples  hp_10h  mage_35  search_4s  sale_20pct  bag_max
   WARRIOR       16      16        0          0           0       15
     ROGUE       16       0        0         14           0       16
    RANGER       16       0        0         15           0        0
      MAGE       16       0       16          0           0        0
    CLERIC       16       0        0          0          15        0
   PALADIN       16       0        0          0          14       15
 cohort  gap_h  growth_spread_pct fastest
 prior8    8.0             3.9284    MAGE
fresh16    8.0             3.5606    MAGE
 prior8   12.0             5.5062 WARRIOR
fresh16   12.0             5.2601 WARRIOR
hero_class  hp_cap_pct  mp_threshold_pct  dex_cap_pct  cha_cap_pct
    CLERIC        0.00              0.00         0.00        90.72
      MAGE        0.00            100.00         0.00         0.00
   PALADIN        0.00              0.00         0.00        90.72
    RANGER        0.00              0.0

## Takeaways

완충 상태에서는 잦은 접속에 마법사, 긴 간격에 전사, 아이템 판매에 성직자가 유리하다. 1~5분의 부분 충전에서는 전사의 보유량·충전량 이점이 커진다. DEX/CHA 특화 일부는 100레벨에도 상한 직전이다. 추가 60,000개는 독립 성장 이벤트 시드의 보조 민감도 분석이며 실제 플레이어 확률이 아니다. 원시 연속 성장 RNG 자료는 제외했다. 운영 앱·DB 변경은 없으며, 배포 전 동적 충전/저장시간 통합 및 기존 성직자 보정 정책 검증이 남는다.
